# Question 1: Online Shopping Platform

Name this Jupyter Notebook as: <br>
`TASK1_<your name>_<class>_<index number>.ipynb`

An online shopping platform stores its data in a SQL database.

The platform needs to store information about products, customers, and order.

Your task is to perform several operations on the database.

For each of the sub-tasks, add a comment statement at the beginning of the code using the hash symbol '#' to indicate the sub-task the program code belongs to.

## Task 1.1 [Modified]

Write Python code to create a SQLite database named online_shop. Ensure that this database is empty before proceeding

Three comma-separated value files are provided, each containing multiple records to be inserted into specific collections within the database. The files are `PRODUCTS.csv`, `CUSTOMERS.csv` and `ORDERS.csv`.

**PRODUCTS.csv:** <br>
`<product_id>,<name>,<model>,<category>,<price>,<stock>`

**CUSTOMERS.csv:** <br>
`<customer_id>,<name>,<email>,<phone>`

**ORDERS.csv:** <br>
`<order_id>,<customer_id>,<product_id>,<quantity>,<order_date>`


Create tables named `PRODUCTS`, `CUSTOMERS`, and `ORDERS`, read the CSV files provided and insert the data from each CSV file into its corresponding table using SQL queries.

The tables should be constructed as follows: <br>
`PRODUCTS (product_id,name,model,category,price,stock)` <br>
`CUSTOMERS (customer_id,name,email,phone)` <br>
`ORDERS (order_id,customer_id,product_id,quantity,order_date)` <br>

Include the necessary primary and foreign keys and ensure that each column has its suitable data type.



Save the SQL command in a file named: <br>
`TASK1_<your name>_<class>_<index number>_tables.sql`

<div style="text-align: right">[3]</div>

In [ ]:
# Task 1.1

import sqlite3
import csv
import os

# if os.path.exists('online_shop.sqlite'):
#     print("file exists")
#     exit()

db = sqlite3.connect('online_shop.sqlite')

db.execute("""CREATE TABLE IF NOT EXISTS PRODUCTS(
    product_id integer PRIMARY KEY,
    name TEXT,
    model TEXT,
    category TEXT,
    price INTEGER,
    stock INTEGER
    )"""
)

db.execute("""CREATE TABLE IF NOT EXISTS CUSTOMERS(
    customer_id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT,
    phone TEXT)"""
)

db.execute("""CREATE TABLE IF NOT EXISTS "ORDERS" (
	order_id	INTEGER,
	customer_id	INTEGER,
    product_id  INTEGER
	quantity    INTEGER,
	order_date	TEXT,
	PRIMARY KEY(order_id, product_id),
	FOREIGN KEY(customer_id) REFERENCES CUSTOMERS(customer_id),
	FOREIGN KEY(product_id) REFERENCES PRODUCTS(product_id)
);"""
)
db.commit()

with open('Resources/TASK1/PRODUCTS.csv', newline='') as f:
    r = csv.DictReader(f)
    for row in r:
        db.execute("""INSERT INTO PRODUCTS VALUES(?, ?, ?, ?, ?, ?)""", 
            (row['product_id'], row['name'], row['model'], 
                  row['category'], row['price'], row['stock']))
    db.commit()
with open('Resources/TASK1/CUSTOMERS.csv', newline='') as f:
    r = csv.DictReader(f)
    for row in r:
        db.execute("""INSERT INTO CUSTOMERS VALUES(?, ?, ?, ?)""",
            (row['customer_id'], row['name'], row['email'], row['phone']))
    db.commit()
with open('Resources/TASK1/ORDERS.csv', newline='') as f:
    r = csv.DictReader(f)
    for row in r:
        db.execute("""INSERT INTO ORDERS VALUES(?, ?, ?, ?, ?)""",
            (row['order_id'], row['customer_id'], row['product_id'],
                        row['quantity'], row['order_date']))
    db.commit()
db.close()
    


## Task 1.2
Add to your program code to retrieve and display all products in the Books category from the `products` table.

Display the data in a readable format.
Run your program.

<div style="text-align:right;">[2]</div>

In [12]:
# Task 1.2

import sqlite3
from pprint import pp

db = sqlite3.connect('online_shop.sqlite')

cx = db.execute("SELECT * FROM PRODUCTS WHERE category = 'Books'")

print('id', 'name', 'model','category','price','stock', sep=', ')
for row in cx:
    print(*row, sep=', ')

db.close()

id, name, model, category, price, stock
11, Novel: The Great Gatsby, Classic Edition, Books, 15, 50
12, Cookbook: The Joy of Cooking, 2020 Edition, Books, 30, 30


## Task 1.3
Add to your program code to find all orders placed by the customer named `Charles`. Assume there is only one customer with this name.

Display the details of these orders from the `orders` table in a readable format.

<div style="text-align:right;">[4]</div>


In [3]:
# Task 1.3

import sqlite3

db = sqlite3.connect('online_shop.sqlite')

charles_id = db.execute("SELECT customer_id FROM CUSTOMERS WHERE name='Charles'").fetchone()[0]
orders = db.execute("SELECT orders.order_id, orders.product_id, orders.quantity, orders.order_date, products.name, products.model, products.category, products.price FROM ORDERS LEFT OUTER JOIN Products on Orders.product_id = Products.product_id where customer_id = ? ", (charles_id,)).fetchall()
print(('order_id', 'product_id', 'qty', 'date', 'name', 'model', 'category', 'price'))
for row in orders:
    print(row)

db.close()


('order_id', 'product_id', 'qty', 'date', 'name', 'model', 'category', 'price')
(4, 5, 1, '2024-04-22', 'Blender', 'BlendPro 500', 'Home Appliances', 80)
(4, 2, 1, '2024-04-22', 'Smartphone', 'Galaxy S21', 'Electronics', 800)
(4, 6, 1, '2024-04-22', 'Smartwatch', 'Apple Watch Series 6', 'Electronics', 400)
(6, 2, 1, '2024-06-18', 'Smartphone', 'Galaxy S21', 'Electronics', 800)
(6, 1, 1, '2024-06-18', 'Laptop', 'XPS 13', 'Electronics', 1200)
(14, 5, 2, '2024-08-15', 'Blender', 'BlendPro 500', 'Home Appliances', 80)
(14, 8, 1, '2024-08-15', 'Vacuum Cleaner', 'Dyson V11', 'Home Appliances', 500)


## Task 1.4
Add to your program code to calculate the revenue generated from all orders.
Display the revenue in a tabular format with headers for Order ID and Revenue.

For each order, compute the revenue as the sum of the price of each product multiplied by the quantity ordered.

Run your program

<div style="text-align: right">[5]</div>

In [9]:
# Task 1.4
import sqlite3

db = sqlite3.connect('online_shop.sqlite')

revenues = db.execute("select orders.order_id, sum(orders.quantity * products.price) as revenue from orders left outer join products on orders.product_id = products.product_id group by orders.order_id").fetchall()
print('id\trevenue')
for row in revenues:
    print(*row, sep='\t')


db.close()

id	revenue
1	800
2	2500
3	230
4	1280
5	350
6	2000
7	2180
8	400
9	5650
10	1700
11	2260
12	4400
13	2400
14	660
15	2250
